# Environnement Video

Sessions video (MiniMax, LTX, Wan, ARP) sans network volume. L'env (`venv_cu130`, CUDA 13) et les modeles sont restaures depuis HF (`rafatribe/Personnal_Loras`).

**Template** : filtre CUDA 13.0 (driver 580 mini), ports HTTP 8188, 8765 (ARP) et 8888, Container Disk ~60 Go, Volume Disk selon les profils (cellule 1).

**Routine**
1. Cellule **1** : boot (env + modeles), quelques minutes.
2. Cellule **3** pour MiniMax / LTX / Wan, **ou** cellule **4** pour ARP (pas les deux en meme temps : meme port 8188).
3. Travailler. Modeles, outputs, inputs, workflows, rendus ARP et Ollama sont sur le Volume Disk : ils survivent a un Stop.
4. Fin de journee : cellule **6** pour zipper, telechargement, puis Terminate.

Les projets ARP en cours (intermediaires, cache, session) ne sont pas gardes : chaque pod repart d'un ARP propre.

Maintenance (rarement) : cellules 7 (mise a jour), 8 (snapshot), 9 (liste des modeles). Cellule 10 : seulement si la cellule 4 dit que l'addon ARP manque.


## 1. Boot de la session (env + modeles)
Choisis les profils dans l'encadre. Le boot verifie le driver et la place avant de telecharger. Un modele absent reste telechargeable via le Model Manager.


In [ ]:
%%bash
# ======================= PROFILS DE MODELES =======================
# Un ou plusieurs, separes par des virgules, sans espace.
#   minimax   MiniMax H3 (fl2va + ref2va) + QwenVL 4B local          ~77 Go
#             + Gemma pour Ollama (cellule 3, une seule fois)        ~20 Go
#   ltx       LTX 2.3 generation : distilled Q5 + upscalers          ~40 Go
#   wan       Wan 2.2 i2v : Remix NSFW + Lightning + SVI             ~46 Go
#   arp       ARP remaster : LTX + Qwen Edit + FlashVSR + SeedVR2    ~87 Go
#   tout      tous les modeles de l'ancien volume video
# Exemples : "minimax"   "ltx,wan"   "arp"
# Volume Disk : additionne les tailles + ~10 Go de marge.
export PROFILS="minimax"
# ===================================================================
export SESSION_ENV="video"

if [ -f /workspace/ComfyUI/main.py ] && [ ! -L /workspace/ComfyUI ] && [ ! -f /workspace/.env_restored ]; then
  echo "Pod d'origine detecte (ComfyUI deja present dans /workspace) : boot inutile."
  exit 0
fi
curl -sL -H "Authorization: Bearer $HF_TOKEN" https://huggingface.co/rafatribe/Personnal_Loras/resolve/main/session/boot.sh | bash

## 2. Suivi du telechargement des modeles
A relancer autant de fois que tu veux pendant le boot.


In [ ]:
%%bash
LOG=/local/fetch_models.log
[ -f "$LOG" ] || LOG=/workspace/fetch_models.log
echo "$(grep -c '^\[' "$LOG" 2>/dev/null || echo 0) fichiers traites"
tail -n 15 "$LOG" 2>/dev/null
df -h / | tail -1 | awk '{print "Container Disk : " $4 " libres sur " $2}'

## 3. Lancement MiniMax / LTX / Wan (Ollama + ComfyUI)
Ollama et le modele Gemma sont installes une seule fois sur le Volume Disk (`/workspace/_ollama_bin`, `/workspace/_ollama`). Mets `OLLAMA = False` si tu n'en as pas besoin.

**Ne pas utiliser "Update All" du Manager** : il casse QwenVL-Mod et huggingface-hub. Passe par la cellule 7.


In [ ]:
# ==================================================================================
# MiniMax / LTX / Wan : Ollama (persistant sur le Volume Disk) puis ComfyUI
# ==================================================================================
import os, sys, time, json, shutil, subprocess, urllib.request

OLLAMA = True   # False pour LTX / Wan si tu n'as pas besoin de Gemma (economise ~20 Go de VRAM et de disque)

COMFY_DIR    = "/workspace/ComfyUI"
VENV_DIR     = "/workspace/venv_cu130"
OLLAMA_BIN   = "/workspace/_ollama_bin"          # binaire Ollama, survit a un Stop
OLLAMA_EXE   = f"{OLLAMA_BIN}/bin/ollama"
OLLAMA_MODEL = "hf.co/HauhauCS/Gemma4-31B-QAT-Uncensored-HauhauCS-Balanced-MTP:Q4_K_M"
OLLAMA_API   = "http://127.0.0.1:11434"
os.environ["OLLAMA_MODELS"]         = "/workspace/_ollama"   # modeles Ollama, survivent a un Stop
os.environ["OLLAMA_HOST"]           = "127.0.0.1:11434"
os.environ["OLLAMA_KEEP_ALIVE"]     = "30s"
os.environ["OLLAMA_CONTEXT_LENGTH"] = "32768"
def api(path, timeout=3):
    with urllib.request.urlopen(f"{OLLAMA_API}{path}", timeout=timeout) as r:
        return json.load(r)

# --- 0. arret des instances precedentes ---------------------------------------
for pat in ("main.py --listen", "arp_launch.py", "ollama serve"):
    subprocess.run(["pkill", "-f", pat], stderr=subprocess.DEVNULL)
time.sleep(3)

if OLLAMA:
    # --- 1. binaire Ollama sur le Volume Disk (installe une seule fois) --------
    os.makedirs(os.environ["OLLAMA_MODELS"], exist_ok=True)
    if not os.path.exists(OLLAMA_EXE):
        print("[ollama] installation sur le Volume Disk (une seule fois)...")
        os.makedirs(OLLAMA_BIN, exist_ok=True)
        subprocess.run("command -v zstd > /dev/null || (apt-get update -qq && apt-get install -y -qq zstd > /dev/null)", shell=True)
        subprocess.run(f"curl -fsSL https://ollama.com/download/ollama-linux-amd64.tar.zst | tar --zstd -x -C {OLLAMA_BIN}", shell=True)
    if not os.path.exists(OLLAMA_EXE):
        print("[ollama] archive indisponible, installation classique (dans le conteneur)...")
        subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, stdout=subprocess.DEVNULL)
        OLLAMA_EXE = shutil.which("ollama")
        if not OLLAMA_EXE:
            sys.exit("[ollama] ECHEC de l'installation, arret.")
    print(f"[ollama] binaire : {OLLAMA_EXE}")

    # --- 2. serveur -----------------------------------------------------------
    log = open("/workspace/ollama.log", "a")
    subprocess.Popen([OLLAMA_EXE, "serve"], stdout=log, stderr=log, env=os.environ, start_new_session=True)
    ready = False
    for _ in range(30):
        try:
            api("/api/tags"); ready = True; break
        except Exception:
            time.sleep(2)
    if not ready:
        sys.exit("[ollama] serveur injoignable, voir /workspace/ollama.log")
    print(f"[ollama] serveur pret (contexte {os.environ['OLLAMA_CONTEXT_LENGTH']}).")

    # --- 3. modele (telecharge une seule fois, garde sur le Volume Disk) -------
    have = [m["name"] for m in api("/api/tags").get("models", [])]
    if not any(OLLAMA_MODEL.split(":")[0] in m for m in have):
        print("[ollama] modele absent, telechargement (~20 Go, une seule fois)...")
        subprocess.run([OLLAMA_EXE, "pull", OLLAMA_MODEL], env=os.environ)
    else:
        print("[ollama] modele deja present.")
    caps = subprocess.run([OLLAMA_EXE, "show", OLLAMA_MODEL],
                          capture_output=True, text=True, env=os.environ).stdout
    print(f"[ollama] vision : {'OK' if 'vision' in caps else 'ABSENTE (mmproj manquant)'}")
else:
    print("[ollama] desactive (OLLAMA = False).")

# --- 4. ComfyUI-Manager : config.ini + patch loopback --------------------------
manager_cfg_dir = f"{COMFY_DIR}/user/__manager"
manager_cfg = f"{manager_cfg_dir}/config.ini"
os.makedirs(manager_cfg_dir, exist_ok=True)
desired_cfg = "[default]\nsecurity_level = weak\nnetwork_mode = personal_cloud\nallow_git_url_install = true\nallow_pip_install = true\n"
if not os.path.exists(manager_cfg) or open(manager_cfg).read() != desired_cfg:
    with open(manager_cfg, "w") as f: f.write(desired_cfg)
    print("[manager] config.ini applique.")
else:
    print("[manager] config.ini OK.")

manager_server_py = f"{COMFY_DIR}/custom_nodes/ComfyUI-Manager/glob/manager_server.py"
if os.path.exists(manager_server_py):
    with open(manager_server_py, "r") as f: content = f.read()
    old_line = "return bool(flag_value) and is_loopback(listen_address)"
    new_line = "return bool(flag_value)  # patched: bypass loopback check"
    if old_line in content:
        with open(manager_server_py, "w") as f: f.write(content.replace(old_line, new_line))
        subprocess.run(["find", f"{COMFY_DIR}/custom_nodes/ComfyUI-Manager",
                        "-name", "__pycache__", "-type", "d", "-exec", "rm", "-rf", "{}", "+"],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        print("[manager] patche (bypass loopback).")
    elif new_line in content:
        print("[manager] deja patche.")
    else:
        print("[manager] WARN: ligne cible introuvable, structure a change.")
else:
    print("[manager] WARN: manager_server.py introuvable.")

# --- 5. ComfyUI -----------------------------------------------------------------
nvidia_libs = "/usr/local/cuda/lib64:/usr/lib/x86_64-linux-gnu/nvidia:/usr/lib/x86_64-linux-gnu"
runpod_id = os.environ.get("RUNPOD_POD_ID")
if runpod_id:
    print(f"\nLIEN RUNPOD : https://{runpod_id}-8188.proxy.runpod.net\n")
run_script = f"""#!/bin/bash
source {VENV_DIR}/bin/activate
export LD_LIBRARY_PATH="{nvidia_libs}:$LD_LIBRARY_PATH"
export PYTHONNOUSERSITE=1
export MPLBACKEND=Agg
cd {COMFY_DIR}
python main.py --listen 0.0.0.0 --port 8188 --enable-cors-header --preview-method auto --reserve-vram 4 --disable-smart-memory
"""
with open("/workspace/run_comfy_cu130.sh", "w") as f: f.write(run_script)
os.chmod("/workspace/run_comfy_cu130.sh", 0o755)
get_ipython().system("/workspace/run_comfy_cu130.sh")

## 4. Lancement ARP (ComfyUI + interface ARP, port 8765)
ComfyUI tourne en arriere-plan (log : `/workspace/comfyui_arp.log`), puis l'interface ARP. Les rendus vont dans `ai-remaster-pipeline/output/reassembled/`.


In [ ]:
# ==================================================================================
# ARP : ComfyUI (arriere-plan, log en direct) puis interface ARP
# ==================================================================================
import os, sys, time, json, subprocess, urllib.request
if not os.path.exists("/workspace/arp_runpod_addon.py"):
    sys.exit("Addon ARP absent : lance d'abord la cellule 10 (annexe), puis relance cette cellule.")

VENV_DIR  = "/workspace/venv_cu130"
COMFY_DIR = "/workspace/ComfyUI"
ARP_DIR   = "/workspace/ai-remaster-pipeline"
PY        = f"{VENV_DIR}/bin/python"

ARP_PORT   = 8765   # port HTTP expose sur le pod pour l'interface ARP
ALLOW_ANY_HOST = True    # le proxy RunPod ne transmet pas le Host attendu par ARP -> 403 sinon
COMFY_PORT = 8188   # si tu ne peux pas exposer 8765 : ARP_PORT=8188 et COMFY_PORT=8189
HF_TOKEN   = os.environ.get("HF_TOKEN", "")

SILENCE_MAX = 600   # abandon seulement si le log ComfyUI ne bouge plus pendant 10 min
MAXLEN      = 600   # lignes geantes coupees a l'affichage (le fichier log reste complet)

# --- 0. arret des instances precedentes + attente que les ports soient libres ---
import socket
def port_busy(p):
    with socket.socket() as so:
        return so.connect_ex(("127.0.0.1", p)) == 0
def stop_all(*sig):
    for pat in ("main.py --listen", "arp_launch.py", "ollama serve"):
        subprocess.run(["pkill", *sig, "-f", pat], stderr=subprocess.DEVNULL)
stop_all()
for i in range(30):
    if not port_busy(COMFY_PORT) and not port_busy(ARP_PORT): break
    if i == 15:
        print("[stop] processus lents a s'arreter -> kill -9")
        stop_all("-9")
    time.sleep(1)
if port_busy(COMFY_PORT) or port_busy(ARP_PORT):
    sys.exit(f"Port {COMFY_PORT} ou {ARP_PORT} toujours occupe. Stoppe la cellule ARP en cours, puis relance.")
print("[stop] ports libres.")

# --- 0b. deps systeme (le conteneur est remis a zero a chaque redemarrage du pod) --
FF_DIR = f"{ARP_DIR}/.cache/tools/ffmpeg"
if not (os.path.exists(f"{FF_DIR}/ffmpeg") and os.path.exists(f"{FF_DIR}/ffprobe")):
    print("[ffmpeg] telechargement d'un build statique (une seule fois, persistant)...")
    os.makedirs(FF_DIR, exist_ok=True)
    url = "https://github.com/BtbN/FFmpeg-Builds/releases/download/latest/ffmpeg-master-latest-linux64-gpl.tar.xz"
    r = subprocess.run(f'cd /tmp && curl -fsSL -o ff.tar.xz "{url}" && tar -xJf ff.tar.xz '
                       f'&& cp ffmpeg-master-latest-linux64-gpl/bin/ffmpeg ffmpeg-master-latest-linux64-gpl/bin/ffprobe "{FF_DIR}/" '
                       f'&& rm -rf ff.tar.xz ffmpeg-master-latest-linux64-gpl', shell=True)
    if r.returncode != 0: sys.exit("[ffmpeg] echec du telechargement.")
print("[ffmpeg]", subprocess.run([f"{FF_DIR}/ffmpeg", "-version"], capture_output=True, text=True).stdout.split("\n")[0][:60])
cv = subprocess.run([PY, "-c", "import cv2"], capture_output=True, text=True)
if cv.returncode != 0:
    print("[apt] OpenCV ne charge pas (libGL manquante apres redemarrage) -> installation...")
    subprocess.run("apt-get update -qq && apt-get install -y -qq libgl1 libglib2.0-0 > /dev/null", shell=True)
    cv = subprocess.run([PY, "-c", "import cv2"], capture_output=True, text=True)
    print("[apt] OpenCV :", "OK" if cv.returncode == 0 else cv.stderr[-300:])

# --- 1. config ARP : ComfyUI externe, gere par nous ----------------------------
with open(f"{ARP_DIR}/.ai_remaster_config.json", "w") as f:
    json.dump({"comfy_dir": COMFY_DIR, "comfy_url": f"http://127.0.0.1:{COMFY_PORT}",
               "comfy_host": "127.0.0.1", "comfy_port": str(COMFY_PORT),
               "comfy_managed_by_arp": "false"}, f, indent=2)

# --- 2. env commun -------------------------------------------------------------
nvidia_libs = "/usr/local/cuda/lib64:/usr/lib/x86_64-linux-gnu/nvidia:/usr/lib/x86_64-linux-gnu"
env = dict(os.environ,
    LD_LIBRARY_PATH=f"{nvidia_libs}:" + os.environ.get("LD_LIBRARY_PATH", ""),
    PYTHONNOUSERSITE="1", MPLBACKEND="Agg", PYTHONUNBUFFERED="1",
    PATH=f"{FF_DIR}:{VENV_DIR}/bin:" + os.environ["PATH"],
    VIRTUAL_ENV=VENV_DIR,
    ARP_HF_CACHE_DIR="/workspace/hf_cache_arp",
    HF_HUB_ENABLE_HF_TRANSFER="1",
    AI_REMASTER_NO_COMFY_AUTOSTART="1",
    AI_REMASTER_NO_BROWSER="1",
    AI_REMASTER_GUI_PORT=str(ARP_PORT),
)
if HF_TOKEN: env["HF_TOKEN"] = HF_TOKEN

# --- 3. ComfyUI en arriere-plan, log affiche en direct --------------------------
cc = subprocess.run(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
                    capture_output=True, text=True).stdout.strip().split("\n")[0]
flags = ["--listen", "0.0.0.0", "--port", str(COMFY_PORT), "--enable-cors-header",
         "--preview-method", "auto", "--reserve-vram", "4", "--disable-smart-memory"]
if float(cc) > 9.0:
    flags.append("--use-pytorch-cross-attention")   # comme ARP le fait sur Blackwell
COMFY_LOG = "/workspace/comfyui_arp.log"
print(f"[comfy] demarrage, log en direct ci-dessous (fichier complet : {COMFY_LOG})")
print("-" * 66)
proc = subprocess.Popen([PY, "-u", "main.py", *flags], cwd=COMFY_DIR, env=env,
                        stdout=open(COMFY_LOG, "w"), stderr=subprocess.STDOUT, start_new_session=True)

state = {"pos": 0, "buf": "", "last": time.time()}
def flush_log(final=False):
    with open(COMFY_LOG, "r", errors="replace") as lf:
        lf.seek(state["pos"]); chunk = lf.read(); state["pos"] = lf.tell()
    if chunk:
        state["last"] = time.time()
    data = state["buf"] + chunk
    lines = data.split("\n")
    state["buf"] = "" if final else lines.pop()       # garde la ligne incomplete pour plus tard
    for line in lines:
        line = line.split("\r")[-1]                    # barres de progression : derniere etape seulement
        print(line if len(line) <= MAXLEN else line[:MAXLEN] + " [...]", flush=True)

def comfy_ready():
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{COMFY_PORT}/system_stats", timeout=2); return True
    except Exception:
        return False

while True:
    flush_log()
    if comfy_ready():
        break
    if proc.poll() is not None:
        time.sleep(1); flush_log(final=True)
        print("-" * 66)
        sys.exit(f"[comfy] ComfyUI s'est arrete (code {proc.returncode}) : l'erreur est juste au-dessus.")
    if time.time() - state["last"] > SILENCE_MAX:
        flush_log(final=True)
        print("-" * 66)
        sys.exit(f"[comfy] plus rien dans le log depuis {SILENCE_MAX // 60} min : bloque apres la derniere ligne ci-dessus.")
    time.sleep(1)
flush_log(final=True)
print("-" * 66)
print("[comfy] pret.")

# --- 4. verif des nodes requis par ARP ----------------------------------------
info = json.load(urllib.request.urlopen(f"http://127.0.0.1:{COMFY_PORT}/object_info", timeout=60))
required = ["ARPLTXVideoOnlyICLoRALoader", "LTXVInpaintPreprocess", "LTXVLaplacianPyramidBlend",
            "LTXAddVideoICLoRAGuideAdvanced", "UnetLoaderGGUF", "VHS_LoadVideo", "VHS_VideoCombine",
            "DeepExColorVideoNode", "ColorMNetVideo"]
missing = [n for n in required if n not in info]
print("[comfy] nodes ARP :", "tous OK" if not missing else f"MANQUANTS -> {missing}")
if missing:
    print("        regarde les 'IMPORT FAILED' : !grep -i 'import failed' -A3", COMFY_LOG)

# --- 5. interface ARP (bind 0.0.0.0 sans modifier le repo) ---------------------
pod = os.environ.get("RUNPOD_POD_ID")
extra_hosts = [f"{pod}-{ARP_PORT}.proxy.runpod.net".lower()] if pod else []
launcher = "/workspace/arp_launch.py"
with open(launcher, "w") as f:
    f.write(f"""import os, sys
sys.path.insert(0, {ARP_DIR!r})
os.chdir({ARP_DIR!r})
import ai_remaster_gui.http_handler as h
h.LOOPBACK_HOSTNAMES.update({extra_hosts!r})
if {ALLOW_ANY_HOST!r}:
    h.Handler.request_host_is_local = lambda self: True
    h.Handler.request_origin_is_local = lambda self: True
sys.path.insert(0, "/workspace")
try:
    import arp_runpod_addon   # Browse web + upload depuis le Mac (cellule 10)
except ImportError:
    print("ATTENTION : /workspace/arp_runpod_addon.py absent, lance la cellule 10")
import ai_remaster_gui.server as s
_orig = s.create_server
s.create_server = lambda host, port: _orig("0.0.0.0", port)
raise SystemExit(s.main())
""")

print("\n" + "=" * 66)
if pod:
    print(f"   ARP      : https://{pod}-{ARP_PORT}.proxy.runpod.net")
    print(f"   ComfyUI  : https://{pod}-{COMFY_PORT}.proxy.runpod.net")
print("   Browse   : explorateur du pod + upload depuis le Mac (glisser-deposer ok)")
print("              + fleche de telechargement sur chaque fichier pour recuperer les rendus.")
print(f"   Sorties  : {ARP_DIR}/output/reassembled/  (Volume Disk, survit a un Stop)")
print(f"   Log Comfy pendant le travail : tail -f {COMFY_LOG}  (dans un terminal)")
print("   Stop     : cellule 5 (ARP + ComfyUI).")
print("=" * 66 + "\n")

with open("/workspace/run_arp.sh", "w") as f:
    f.write("#!/bin/bash\n" + "".join(f'export {k}="{v}"\n' for k, v in env.items()
            if k.startswith(("ARP_", "AI_REMASTER", "HF_", "LD_LIBRARY", "PYTHONNOUSERSITE", "MPLBACKEND", "VIRTUAL_ENV")))
            + f'export PATH="{FF_DIR}:{VENV_DIR}/bin:$PATH"\nexec {PY} {launcher}\n')
os.chmod("/workspace/run_arp.sh", 0o755)
get_ipython().system("/workspace/run_arp.sh")

## 5. Tout arreter (ARP, ComfyUI, Ollama)
A faire avant un snapshot (cellule 8), ou pour passer de la cellule 3 a la cellule 4.


In [ ]:
%%bash
pkill -f arp_launch.py && echo "Interface ARP arretee."
if pkill -f "main.py --listen"; then sleep 3; echo "ComfyUI arrete."; else echo "ComfyUI ne tournait pas."; fi
pkill -f "ollama serve" && echo "Ollama arrete."
true

## 6. Zipper les outputs
Un zip pour ComfyUI, un pour les rendus ARP. Ils apparaissent dans `/workspace`, panneau de gauche de Jupyter : clic droit > Download.


In [ ]:
%%bash
STAMP=$(date +%Y%m%d-%H%M)
DEST=/workspace; [ -d /local ] && DEST=/local
mkzip() {   # mkzip <dossier> <nom>
  [ -n "$(ls -A "$1" 2>/dev/null)" ] || { echo "$1 : vide, rien a zipper."; return; }
  (cd "$1" && python3 -m zipfile -c "$DEST/$2" .)
  [ "$DEST" = /local ] && ln -sf "/local/$2" "/workspace/$2"
  ls -lhL "/workspace/$2"
}
mkzip /workspace/ComfyUI/output "outputs_$STAMP.zip"
mkzip /workspace/ai-remaster-pipeline/output/reassembled "arp_rendus_$STAMP.zip"

## 7. Mettre a jour ComfyUI et les custom nodes
Dependances installees dans `venv_cu130`. Les nodes modifies localement et les nodes ARP (sans git) sont laisses tels quels. huggingface-hub est re-epingle sous 2.0 a la fin. Ensuite : relance (cellule 5 puis 3 ou 4) et teste tes workflows avant tout snapshot.


In [ ]:
%%bash
export SESSION_ENV=video VENV=venv_cu130
PIP=/workspace/$VENV/bin/pip
cd /workspace/ComfyUI || exit 1

echo "=== ComfyUI ==="
DELETED=$(git ls-files --deleted)
[ -n "$DELETED" ] && git checkout -q -- $DELETED
if [ -n "$(git status --porcelain --untracked-files=no)" ]; then
  echo "!! ComfyUI a des modifications locales, mise a jour sautee :"
  git status --short --untracked-files=no
else
  B=$(git rev-parse --short HEAD)
  git fetch -q origin && git checkout -q master && git pull -q --ff-only origin master
  A=$(git rev-parse --short HEAD)
  echo "ComfyUI : $B -> $A"
  [ "$B" != "$A" ] && $PIP install -q -r requirements.txt
fi

echo ""
echo "=== Custom nodes ==="
for d in custom_nodes/*/; do
  n=$(basename "$d")
  [ -d "$d/.git" ] || { echo "  -- $n : pas un depot git (nodes ARP), ignore"; continue; }
  if [ -n "$(git -C "$d" status --porcelain --untracked-files=no)" ]; then
    echo "  !! $n : modifie localement, laisse tel quel"; continue
  fi
  B=$(git -C "$d" rev-parse --short HEAD)
  git -C "$d" pull -q --ff-only 2>/dev/null || { echo "  !! $n : mise a jour impossible"; continue; }
  A=$(git -C "$d" rev-parse --short HEAD)
  if [ "$B" != "$A" ]; then
    echo "  OK $n : $B -> $A"
    [ -f "$d/requirements.txt" ] && $PIP install -q -r "$d/requirements.txt"
  fi
done

echo ""
echo "=== Versions epinglees ==="
$PIP install -q "huggingface-hub>=1.31,<2.0" && echo "huggingface-hub : $($PIP show huggingface-hub | grep ^Version | cut -d' ' -f2) (doit rester < 2.0 pour diffusers)"
echo ""
echo "Termine. Relance ComfyUI (cellule 5 puis cellule 3 ou 4), puis teste tes workflows (QwenVL en premier)."

## 8. Snapshot de l'env (nouvelle base pour les prochains pods)
Uniquement apres avoir teste tes workflows. Arrete tout avant (cellule 5). Les projets ARP, caches et sorties ne sont pas inclus. L'ancien snapshot reste actif tant que le nouveau n'est pas entierement envoye. Evite d'en faire plusieurs le meme jour (place HF liberee sous 36 h).


In [ ]:
%%bash
export SESSION_ENV=video VENV=venv_cu130
export PATH=/workspace/$VENV/bin:$PATH
pgrep -f "main.py --listen" > /dev/null && { echo "ComfyUI tourne encore : lance d'abord la cellule 5."; exit 1; }
curl -sL -H "Authorization: Bearer $HF_TOKEN" https://huggingface.co/rafatribe/Personnal_Loras/resolve/main/session/snapshot_env.sh | bash

## 9. Garder les modeles ajoutes pendant la session (optionnel)
Met a jour la liste du profil "tout" (video) a partir des modeles presents sur le disque.


In [ ]:
%%bash
export SESSION_ENV=video VENV=venv_cu130
export PATH=/workspace/$VENV/bin:$PATH
LISTE=session_models_video.txt
cd /workspace/ComfyUI/models || exit 1
find -L . -type f \( -name "*.safetensors" -o -name "*.ckpt" -o -name "*.pt" -o -name "*.pth" -o -name "*.gguf" -o -name "*.bin" \) -size +0 \
  | sed 's#^\./##' | grep -v '^sam3/sam3.pt$' | grep -v '/\.cache/' | sort > /tmp/session_models.txt
echo "$(wc -l < /tmp/session_models.txt) modeles dans la liste $LISTE (profil \"tout\")"
hf upload rafatribe/Personnal_Loras /tmp/session_models.txt "session/$LISTE" --commit-message "Liste modeles $(date +%F)"

## 10. Annexe : addon ARP (seulement s'il manque)
Ecrit `/workspace/arp_runpod_addon.py` (explorateur web + upload depuis le Mac dans ARP). Normalement deja fourni par le snapshot.


In [ ]:
%%writefile /workspace/arp_runpod_addon.py
# ==================================================================================
# ANNEXE - ADDON RUNPOD : Browse web + upload Mac (ecrit /workspace/arp_runpod_addon.py)
# ==================================================================================
# ARP RunPod addon : remplace les selecteurs de fichiers natifs (zenity/kdialog, absents
# sur un pod) par une fenetre web : navigation dans les dossiers du pod, upload depuis le Mac
# (par morceaux de 32 Mo pour passer la limite de 100 Mo du proxy RunPod) et telechargement.
# Importe par /workspace/arp_launch.py AVANT ai_remaster_gui.server. Ne modifie pas le repo.
import json, os, threading, time, uuid
from pathlib import Path
from urllib.parse import urlparse, parse_qs

import ai_remaster_gui.file_dialogs as fd
import ai_remaster_gui.http_handler as hh
from ai_remaster_gui.config import ROOT, STATIC_DIR
from ai_remaster_gui.paths import rel

BROWSE_ROOT = Path(os.environ.get("ARP_RP_ROOT", "/workspace")).resolve()
DEFAULT_DIR = ROOT / "input"
WAIT_SECONDS = 85          # le proxy RunPod coupe une requete au bout de ~100 s
SAVE_KINDS = {"save", "save_image", "project_save"}

_lock = threading.Lock()
_preselect: list[str] | None = None          # choix fait AVANT l'appel (Browse principal)
_pending: dict | None = None                 # dialogue demande par le serveur
_answers: dict[str, list[str] | None] = {}
_events: dict[str, threading.Event] = {}


def _inside(path: Path) -> Path:
    p = path.expanduser().resolve()
    p.relative_to(BROWSE_ROOT)               # ValueError si hors de /workspace
    return p


def _ask_browser(kind: str, initial: Path, multiple: bool) -> list[str] | None:
    """Prend la preselection si elle existe, sinon ouvre la fenetre web et attend la reponse."""
    global _preselect, _pending
    with _lock:
        if _preselect is not None:
            chosen, _preselect = _preselect, None
            return chosen
        rid = uuid.uuid4().hex
        start = initial if initial.is_dir() else initial.parent
        try:
            start = _inside(start)
        except ValueError:
            start = DEFAULT_DIR
        _pending = {"id": rid, "kind": kind, "multiple": multiple, "dir": str(start),
                    "name": initial.name if kind in SAVE_KINDS and not initial.is_dir() else ""}
        ev = _events[rid] = threading.Event()
    ev.wait(WAIT_SECONDS)
    with _lock:
        if _pending and _pending["id"] == rid:
            _pending = None
        _events.pop(rid, None)
        return _answers.pop(rid, None)


def browse_path_linux(kind: str, initial: Path) -> str:
    chosen = _ask_browser(kind, initial, False)
    if not chosen:
        fd.state.APP.log.append("Browse cancelled.")
        return ""
    fd.state.APP.log.append(f"Browse selected: {chosen[0]}")
    return rel(Path(chosen[0]))


def browse_source_paths_linux(initial: Path) -> list[str]:
    chosen = _ask_browser("source", initial, True)
    if not chosen:
        fd.state.APP.log.append("Browse cancelled.")
        return []
    fd.state.APP.log.append(f"Browse selected {len(chosen)} source item(s).")
    return [rel(Path(p)) for p in chosen]


fd.browse_path_linux = browse_path_linux
fd.browse_source_paths_linux = browse_source_paths_linux

# ------------------------------------------------------------------------------- HTTP
_orig_get, _orig_post = hh.Handler.do_GET, hh.Handler.do_POST


def _listing(path: Path) -> dict:
    dirs, files = [], []
    for e in sorted(path.iterdir(), key=lambda e: e.name.lower()):
        if e.name.startswith(".") or e.name.endswith(".rp-part"):
            continue
        try:
            if e.is_dir():
                dirs.append(e.name)
            else:
                st = e.stat()
                files.append({"name": e.name, "size": st.st_size, "mtime": st.st_mtime})
        except OSError:
            pass
    parent = str(path.parent) if path != BROWSE_ROOT else ""
    return {"ok": True, "path": str(path), "parent": parent, "dirs": dirs, "files": files,
            "shortcuts": {"input": str(DEFAULT_DIR), "output": str(ROOT / "output"),
                          "intermediate": str(ROOT / "intermediate"), "workspace": str(BROWSE_ROOT)}}


def do_GET(self):
    u = urlparse(self.path)
    q = {k: v[0] for k, v in parse_qs(u.query).items()}
    if not self.request_host_is_local():
        return _orig_get(self)
    try:
        if u.path == "/":
            html = (STATIC_DIR / "index.html").read_text(encoding="utf-8")
            html = html.replace("</body>", '<script src="/rp-addon.js"></script>\n</body>')
            return self.send_text(html, "text/html; charset=utf-8")
        if u.path == "/rp-addon.js":
            return self.send_text(ADDON_JS, "application/javascript; charset=utf-8")
        if u.path == "/api/rp-ls":
            return self.send_json(_listing(_inside(Path(q.get("path") or DEFAULT_DIR))))
        if u.path == "/api/rp-dialog":
            with _lock:
                return self.send_json({"ok": True, "pending": _pending})
        if u.path == "/api/rp-download":
            p = _inside(Path(q["path"]))
            self.send_response(200)
            self.send_header("Content-Type", "application/octet-stream")
            self.send_header("Content-Length", str(p.stat().st_size))
            self.send_header("Content-Disposition", f'attachment; filename="{p.name}"')
            self.end_headers()
            with p.open("rb") as f:
                while chunk := f.read(4 << 20):
                    self.wfile.write(chunk)
            return
    except (ValueError, KeyError, OSError) as exc:
        return self.send_json({"ok": False, "error": f"{type(exc).__name__}: {exc}"})
    return _orig_get(self)


def do_POST(self):
    global _preselect, _pending
    u = urlparse(self.path)
    if not u.path.startswith("/api/rp-") or not (self.request_host_is_local() and self.request_origin_is_local()):
        return _orig_post(self)
    q = {k: v[0] for k, v in parse_qs(u.query).items()}
    try:
        if u.path == "/api/rp-upload":
            folder = _inside(Path(q.get("dir") or DEFAULT_DIR))
            folder.mkdir(parents=True, exist_ok=True)
            name = Path(q["name"]).name
            offset, total = int(q["offset"]), int(q["total"])
            part = folder / (name + ".rp-part")
            length = int(self.headers.get("Content-Length", "0"))
            with part.open("r+b" if offset and part.exists() else "wb") as f:
                f.seek(offset)
                left = length
                while left:
                    buf = self.rfile.read(min(left, 4 << 20))
                    if not buf:
                        raise OSError("upload interrompu")
                    f.write(buf)
                    left -= len(buf)
            done = offset + length >= total
            if done:
                os.replace(part, folder / name)
            return self.send_json({"ok": True, "done": done, "path": str(folder / name)})
        data = self.read_json()
        if u.path == "/api/rp-preselect":
            paths = [str(_inside(Path(p))) for p in data.get("paths", [])]
            with _lock:
                _preselect = paths or None
            return self.send_json({"ok": True})
        if u.path == "/api/rp-answer":
            rid = data.get("id", "")
            paths = data.get("paths")
            if paths:
                paths = [str(_inside(Path(p))) for p in paths]
            with _lock:
                _answers[rid] = paths or None
                if _pending and _pending["id"] == rid:
                    _pending = None
                ev = _events.get(rid)
            if ev:
                ev.set()
            return self.send_json({"ok": True})
        if u.path == "/api/rp-mkdir":
            p = _inside(Path(data["path"]))
            p.mkdir(parents=True, exist_ok=True)
            return self.send_json({"ok": True, "path": str(p)})
    except (ValueError, KeyError, OSError) as exc:
        return self.send_json({"ok": False, "error": f"{type(exc).__name__}: {exc}"})
    self.send_error(404)


hh.Handler.do_GET = do_GET
hh.Handler.do_POST = do_POST

ADDON_JS = r"""
(() => {
const CHUNK = 32 * 1024 * 1024;
const fmt = n => n > 1e9 ? (n / 1e9).toFixed(2) + ' Go' : n > 1e6 ? (n / 1e6).toFixed(1) + ' Mo' : Math.max(1, Math.round(n / 1e3)) + ' Ko';
const esc = s => String(s).replace(/[&<>"']/g, c => ({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[c]));
const MEDIA = /\.(mp4|mov|mkv|avi|webm|m4v|png|bmp|jpe?g|dpx|webp|tiff?|arp|json)$/i;

const css = document.createElement('style');
css.textContent = `
#rpModal{position:fixed;inset:0;background:rgba(0,0,0,.7);z-index:99999;display:flex;align-items:center;justify-content:center;font:14px system-ui,sans-serif}
#rpModal .box{background:#15181c;color:#ddd;border:1px solid #333;border-radius:10px;width:min(900px,94vw);height:min(720px,90vh);display:flex;flex-direction:column;padding:16px;gap:10px}
#rpModal h3{margin:0;font-size:17px}
#rpModal .bar{display:flex;gap:6px;flex-wrap:wrap;align-items:center}
#rpModal button{background:#252a31;color:#ddd;border:1px solid #3a4049;border-radius:6px;padding:6px 12px;cursor:pointer}
#rpModal button:hover{background:#2f353e}
#rpModal button.pri{background:#2d6cdf;border-color:#2d6cdf;color:#fff}
#rpModal button:disabled{opacity:.4;cursor:default}
#rpModal .path{font-family:monospace;font-size:12px;color:#9aa;word-break:break-all}
#rpModal .list{flex:1;overflow:auto;border:1px solid #2a2f36;border-radius:6px;background:#0f1114}
#rpModal .row{display:flex;align-items:center;gap:8px;padding:6px 10px;border-bottom:1px solid #1d2127;cursor:pointer;user-select:none}
#rpModal .row:hover{background:#1b2027}
#rpModal .row.sel{background:#1e3a66}
#rpModal .row .n{flex:1;overflow:hidden;text-overflow:ellipsis;white-space:nowrap}
#rpModal .row .s{color:#888;font-size:12px;min-width:80px;text-align:right}
#rpModal .row a{color:#7ab;text-decoration:none;padding:0 4px}
#rpModal .drop{border:2px dashed #3a4049;border-radius:8px;padding:10px;text-align:center;color:#999}
#rpModal .drop.over{border-color:#2d6cdf;color:#fff}
#rpModal .prog{height:8px;background:#222;border-radius:4px;overflow:hidden}
#rpModal .prog div{height:100%;width:0;background:#2d6cdf}
#rpModal input[type=text]{flex:1;background:#0f1114;color:#ddd;border:1px solid #3a4049;border-radius:6px;padding:6px}
`;
document.head.appendChild(css);

async function getJ(u){ const r = await fetch(u); return r.json(); }
async function postJ(u, d){ const r = await fetch(u,{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify(d)}); return r.json(); }

// opts: {kind, multiple, dir, name}  ->  Promise<string[]|null>
function openDialog(opts) {
  return new Promise(async resolve => {
    document.getElementById('rpModal')?.remove();
    const kind = opts.kind || 'file';
    const isFolder = kind === 'folder', isSave = ['save','save_image','project_save'].includes(kind);
    const title = isFolder ? 'Choisir un dossier' : isSave ? 'Enregistrer sous' : 'Choisir ' + (opts.multiple ? 'une video ou des images' : 'un fichier');
    const m = document.createElement('div');
    m.id = 'rpModal';
    m.innerHTML = `<div class="box">
      <h3>${title} <span style="color:#777;font-weight:normal;font-size:12px">(pod RunPod)</span></h3>
      <div class="bar"><button data-a="up">&uarr; Parent</button>
        <button data-sc="input">input/</button><button data-sc="output">output/</button>
        <button data-sc="intermediate">intermediate/</button><button data-sc="workspace">/workspace</button>
        <button data-a="mkdir">+ Dossier</button></div>
      <div class="path"></div>
      <div class="list"></div>
      <div class="drop">Glisse des fichiers ici ou <button data-a="pick">Uploader depuis le Mac</button>
        <input type="file" multiple style="display:none"><div class="info" style="margin-top:6px;font-size:12px"></div>
        <div class="prog" style="display:none"><div></div></div></div>
      ${isSave ? '<div class="bar"><span>Nom :</span><input type="text" class="fname"></div>' : ''}
      <div class="bar" style="justify-content:flex-end"><button data-a="cancel">Annuler</button><button class="pri" data-a="ok" disabled>Choisir</button></div>
    </div>`;
    document.body.appendChild(m);
    const $ = s => m.querySelector(s);
    let cur = null, shortcuts = {}, sel = new Set(), uploading = false;
    const fname = $('.fname'); if (fname) fname.value = opts.name || '';
    const done = v => { m.remove(); resolve(v); };
    const updOk = () => {
      const ok = $('[data-a=ok]');
      if (isFolder) { ok.disabled = !cur; ok.textContent = 'Choisir ce dossier'; }
      else if (isSave) { ok.disabled = !(fname.value.trim()); ok.textContent = 'Enregistrer ici'; }
      else { ok.disabled = !sel.size; ok.textContent = sel.size > 1 ? `Choisir (${sel.size})` : 'Choisir'; }
      ok.disabled = ok.disabled || uploading;
    };
    if (fname) fname.oninput = updOk;
    async function load(dir) {
      const r = await getJ('/api/rp-ls?path=' + encodeURIComponent(dir || ''));
      if (!r.ok) return alert(r.error);
      cur = r.path; shortcuts = r.shortcuts; sel.clear();
      $('.path').textContent = cur;
      $('[data-a=up]').disabled = !r.parent;
      $('[data-a=up]').onclick = () => load(r.parent);
      const L = $('.list'); L.innerHTML = '';
      r.dirs.forEach(d => { const e = document.createElement('div'); e.className = 'row';
        e.innerHTML = `<span>&#128193;</span><span class="n">${esc(d)}</span>`; e.onclick = () => load(cur + '/' + d); L.appendChild(e); });
      r.files.forEach(f => { const full = cur + '/' + f.name; const e = document.createElement('div'); e.className = 'row';
        e.style.opacity = isFolder ? .45 : (MEDIA.test(f.name) ? 1 : .6);
        e.innerHTML = `<span>&#128196;</span><span class="n">${esc(f.name)}</span><span class="s">${fmt(f.size)}</span>
          <a href="/api/rp-download?path=${encodeURIComponent(full)}" title="Telecharger sur le Mac" onclick="event.stopPropagation()">&#11015;</a>`;
        if (!isFolder) {
          e.onclick = ev => {
            if (isSave) { fname.value = f.name; return updOk(); }
            if (!(opts.multiple && (ev.metaKey || ev.ctrlKey || ev.shiftKey))) { sel.clear(); L.querySelectorAll('.sel').forEach(x => x.classList.remove('sel')); }
            sel.has(full) ? (sel.delete(full), e.classList.remove('sel')) : (sel.add(full), e.classList.add('sel')); updOk(); };
          e.ondblclick = () => { if (!isSave) done([full]); };
        }
        L.appendChild(e); });
      updOk();
    }
    async function upload(files) {
      if (!files.length) return;
      uploading = true; updOk();
      const info = $('.info'), bar = $('.prog'), fill = $('.prog div');
      bar.style.display = 'block';
      const total = [...files].reduce((a, f) => a + f.size, 0); let sent = 0; const t0 = Date.now(); const up = [];
      for (const f of files) {
        for (let off = 0; off < f.size || (off === 0 && f.size === 0); off += CHUNK) {
          const blob = f.slice(off, off + CHUNK);
          let r = null;
          for (let tryN = 0; tryN < 4 && !(r && r.ok); tryN++) {
            try {
              const res = await fetch(`/api/rp-upload?dir=${encodeURIComponent(cur)}&name=${encodeURIComponent(f.name)}&offset=${off}&total=${f.size}`, {method:'POST', body: blob});
              r = await res.json();
            } catch (e) { r = {ok:false, error:String(e)}; await new Promise(z => setTimeout(z, 2000)); }
          }
          if (!r.ok) { uploading = false; updOk(); return alert('Upload echoue : ' + r.error); }
          sent += blob.size;
          const s = (Date.now() - t0) / 1000;
          fill.style.width = (100 * sent / Math.max(1, total)) + '%';
          info.textContent = `${f.name} : ${fmt(sent)} / ${fmt(total)}  (${fmt(sent / Math.max(s, .1))}/s)`;
          if (f.size === 0) break;
        }
        up.push(cur + '/' + f.name);
      }
      info.textContent = `Termine : ${up.length} fichier(s) dans ${cur}`;
      uploading = false;
      await load(cur);
      if (!isFolder && !isSave) {
        const keep = opts.multiple ? up : up.slice(-1);
        $('.list').querySelectorAll('.row').forEach(row => {
          const n = row.querySelector('.n')?.textContent; if (keep.includes(cur + '/' + n)) { sel.add(cur + '/' + n); row.classList.add('sel'); } });
        updOk();
      }
    }
    m.querySelectorAll('[data-sc]').forEach(b => b.onclick = () => load(shortcuts[b.dataset.sc]));
    $('[data-a=pick]').onclick = () => $('input[type=file]').click();
    $('input[type=file]').onchange = e => upload(e.target.files);
    $('[data-a=mkdir]').onclick = async () => { const n = prompt('Nom du nouveau dossier'); if (!n) return;
      const r = await postJ('/api/rp-mkdir', {path: cur + '/' + n}); r.ok ? load(r.path) : alert(r.error); };
    const drop = $('.drop');
    m.addEventListener('dragover', e => { e.preventDefault(); drop.classList.add('over'); });
    m.addEventListener('dragleave', () => drop.classList.remove('over'));
    m.addEventListener('drop', e => { e.preventDefault(); drop.classList.remove('over'); upload(e.dataTransfer.files); });
    $('[data-a=cancel]').onclick = () => { if (!uploading || confirm('Upload en cours, annuler ?')) done(null); };
    $('[data-a=ok]').onclick = () => done(isFolder ? [cur] : isSave ? [cur + '/' + fname.value.trim()] : [...sel]);
    await load(opts.dir || '');
  });
}

// 1) Browse principaux : choix d'abord (upload possible sans limite de temps), puis appel ARP
const origGlobal = window.browseGlobalSource, origField = window.browseField;
window.browseGlobalSource = async function () {
  const paths = await openDialog({kind: 'source', multiple: true});
  if (!paths || !paths.length) return;
  await postJ('/api/rp-preselect', {paths});
  return origGlobal.apply(this, arguments);
};
window.browseField = async function (stageKey, fieldKey, kind) {
  const paths = await openDialog({kind, multiple: false});
  if (!paths || !paths.length) return;
  await postJ('/api/rp-preselect', {paths});
  return origField.apply(this, arguments);
};

// 2) Autres selecteurs declenches cote serveur (projet, images de reference...) : polling
let busy = false;
setInterval(async () => {
  if (busy || document.getElementById('rpModal')) return;
  try {
    const r = await getJ('/api/rp-dialog');
    if (!r.pending) return;
    busy = true;
    const p = r.pending;
    const paths = await openDialog({kind: p.kind, multiple: p.multiple, dir: p.dir, name: p.name});
    await postJ('/api/rp-answer', {id: p.id, paths});
  } catch (e) {} finally { busy = false; }
}, 1000);
})();
"""